# 02. Data Cleaning & Feature Engineering

## Objective
The goal of this notebook is to execute a rigorous data cleaning pipeline to prepare raw transactions for RFM analysis.

### Cleaning Steps:
1. Remove exact duplicate rows.
2. Filter out missing `CustomerID` records.
3. Separate cancelled transactions (`InvoiceNo` starting with 'C').
4. Filter out non-positive quantities (`Quantity <= 0`).
5. Filter out non-positive unit prices (`UnitPrice <= 0`).
6. Calculate transaction-level `Revenue = Quantity * UnitPrice`.
7. Save cleaned dataset to `data/processed/cleaned_transactions.csv`.

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print("Pandas loaded.")

### Step 2: Load Raw Data

In [ ]:
df = pd.read_excel('../data/raw/Online Retail.xlsx')
initial_count = len(df)
print(f"Loaded raw dataset with {initial_count:,} rows.")

### Step 3: Remove Exact Duplicate Rows
Duplicate transactions occur due to system logging errors or repeated form submissions.

In [ ]:
duplicates = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {duplicates:,} duplicate rows. Remaining: {len(df):,}")

### Step 4: Handle Missing CustomerID
RFM analysis requires tracking individual customer identity. Unauthenticated guest checkouts cannot be assigned to an RFM profile.

In [ ]:
missing_cust = df['CustomerID'].isnull().sum()
df = df.dropna(subset=['CustomerID'])
df['CustomerID'] = df['CustomerID'].astype(int)
print(f"Removed {missing_cust:,} rows with missing CustomerID. Remaining: {len(df):,}")

### Step 5: Identify & Remove Cancelled Transactions
Invoices beginning with letter 'C' represent order cancellations and refunds.

In [ ]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
cancellations_mask = df['InvoiceNo'].str.startswith('C')
cancellations_count = cancellations_mask.sum()
df = df[~cancellations_mask]
print(f"Removed {cancellations_count:,} cancelled transactions. Remaining: {len(df):,}")

### Step 6: Filter Out Non-Positive Quantities (Quantity <= 0)
Completed purchase records must have a strictly positive item quantity.

In [ ]:
invalid_qty_count = (df['Quantity'] <= 0).sum()
df = df[df['Quantity'] > 0]
print(f"Removed {invalid_qty_count:,} non-positive quantity rows. Remaining: {len(df):,}")

### Step 7: Filter Out Invalid Unit Prices (UnitPrice <= 0)
Transactions with zero or negative prices represent administrative adjustments or data entries.

In [ ]:
invalid_price_count = (df['UnitPrice'] <= 0).sum()
df = df[df['UnitPrice'] > 0]
print(f"Removed {invalid_price_count:,} invalid unit price rows. Remaining: {len(df):,}")

### Step 8: Ensure Correct Datetime Formatting
Convert `InvoiceDate` to pandas Datetime object.

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"InvoiceDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

### Step 9: Feature Engineering — Calculate Revenue
Calculate transaction revenue: `Revenue = Quantity * UnitPrice`.

In [ ]:
df['Revenue'] = (df['Quantity'] * df['UnitPrice']).round(2)
df[['InvoiceNo', 'Quantity', 'UnitPrice', 'Revenue', 'CustomerID']].head()

### Step 10: Extract Temporal Features
Extract `InvoiceYear`, `InvoiceMonth`, `InvoiceDay`, `InvoiceHour`, and `InvoiceMonthYear`.

In [ ]:
df['InvoiceYear'] = df['InvoiceDate'].dt.year
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceDay'] = df['InvoiceDate'].dt.day
df['InvoiceHour'] = df['InvoiceDate'].dt.hour
df['InvoiceMonthYear'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df[['InvoiceDate', 'InvoiceMonthYear', 'Revenue']].head()

### Step 11: Inspect Cleaned Summary Statistics
Verify that non-positive quantities and prices are completely eliminated.

In [ ]:
df[['Quantity', 'UnitPrice', 'Revenue']].describe()

### Step 12: Save Cleaned Transactions Dataset
Export cleaned transactions to `../data/processed/cleaned_transactions.csv`.

In [ ]:
output_path = '../data/processed/cleaned_transactions.csv'
df.to_csv(output_path, index=False)
final_count = len(df)
retention_pct = (final_count / initial_count) * 100
print(f"Cleaned dataset saved successfully to {output_path}")
print(f"Final Cleaned Rows: {final_count:,} ({retention_pct:.2f}% retained)")